# Model Building

## 1. Importing Libraries

In [75]:
import pandas as pd
import numpy as np

## 2. Load Train and Test Data

In [76]:
x_train = pd.read_csv("../data/x_train.csv")

In [77]:
x_test = pd.read_csv("../data/x_test.csv")

In [78]:
y_train = pd.read_csv("../data/y_train.csv").squeeze()

In [79]:
y_test = pd.read_csv("../data/y_test.csv").squeeze()

In [80]:
x_train.shape

(491, 13)

In [81]:
x_test.shape

(123, 13)

In [82]:
y_train.shape

(491,)

In [83]:
y_test.shape

(123,)

## 3. Model training

In [84]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

### 3.1 Logistic Regression

In [85]:
from sklearn.linear_model import LogisticRegression

In [124]:
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [125]:
y_pred = model.predict(x_test)

In [126]:
accuracy_score(y_test, y_pred)

0.8617886178861789

In [127]:
confusion_matrix(y_test, y_pred)

array([[22, 16],
       [ 1, 84]])

In [128]:
classification_report(y_test, y_pred)

'              precision    recall  f1-score   support\n\n           0       0.96      0.58      0.72        38\n           1       0.84      0.99      0.91        85\n\n    accuracy                           0.86       123\n   macro avg       0.90      0.78      0.81       123\nweighted avg       0.88      0.86      0.85       123\n'

## Baseline Model Results — Logistic Regression

- **Accuracy: 86%** — the model correctly predicted loan status for 106 out of 123 test applicants.

- **Confusion Matrix:**
  - Rejected (0): correctly identified 22 out of 38 actual rejections (missed 16)
  - Approved (1): correctly identified 84 out of 85 actual approvals (missed only 1)

- **Main Strength:** extremely good at identifying approved loans — 99% recall for class 1 means it almost never misses a genuine approval. If the business priority is to not reject good applicants, this model does that well.

- **Main Weakness:** poor at catching rejections — only 58% recall for class 0 means it misses 16 out of 38 actual rejections, incorrectly approving them. For a loan prediction model, this is a real concern — approving loans that should be rejected is a costly mistake for a bank.

In [93]:
coef_df = pd.DataFrame({
    "Feature": x_train.columns,
    "Coefficient": model.coef_[0]
})

coef_df.sort_values(
    by="Coefficient",
    key=abs,
    ascending=False
)

,Feature,Coefficient
9,Credit_History,3.167054
11,Property_Area_Semiurban,0.685137
1,Married,0.502156
3,Education,0.374347
10,Credit_History_Missing,-0.320725
4,Self_Employed,-0.198173
0,Gender,-0.163436
12,Property_Area_Urban,0.112174
6,CoapplicantIncome,-0.107266
2,Dependents,0.093117


### 3.2 Decision Tree

In [94]:
from sklearn.tree import DecisionTreeClassifier

In [95]:
dt = DecisionTreeClassifier(random_state=42)

In [96]:
dt.fit(x_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [97]:
y_pred_dt = dt.predict(x_test)

In [98]:
accuracy_score(y_test, y_pred_dt)

0.7723577235772358

In [99]:
confusion_matrix(y_test, y_pred_dt)

array([[26, 12],
       [16, 69]])

In [100]:
classification_report(y_test, y_pred_dt)

'              precision    recall  f1-score   support\n\n           0       0.62      0.68      0.65        38\n           1       0.85      0.81      0.83        85\n\n    accuracy                           0.77       123\n   macro avg       0.74      0.75      0.74       123\nweighted avg       0.78      0.77      0.78       123\n'

In [101]:
dt.score(x_train, y_train)

1.0

In [102]:
dt.score(x_test, y_test)

0.7723577235772358

#### Important finding:
- Train Accuracy = 100%
- Test Accuracy = 77.24%
- Strong overfitting observed.

### 3.3 Random Forest Regressor

In [103]:
from sklearn.ensemble import RandomForestClassifier

In [104]:
rf = RandomForestClassifier(random_state=42)

In [105]:
rf.fit(x_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [106]:
y_pred_rf = rf.predict(x_test)

In [107]:
accuracy_score(y_test, y_pred_rf)

0.8292682926829268

In [109]:
confusion_matrix(y_test, y_pred_rf)

array([[23, 15],
       [ 6, 79]])

In [110]:
classification_report(y_test, y_pred_rf)

'              precision    recall  f1-score   support\n\n           0       0.79      0.61      0.69        38\n           1       0.84      0.93      0.88        85\n\n    accuracy                           0.83       123\n   macro avg       0.82      0.77      0.78       123\nweighted avg       0.83      0.83      0.82       123\n'

In [111]:
 rf.score(x_train, y_train)

1.0

In [112]:
rf.score(x_test, y_test)

0.8292682926829268

#### Finding:
- Reduced overfitting compared to a single tree,
- but still failed to outperform Logistic Regression.

### 3.4 Logistic Regression Hyperparameter Tuning

In [122]:
results = []

for c in [0.01, 0.1, 1, 10, 100]:

    lr = LogisticRegression(
        C=c,
        max_iter=1000,
        random_state=42
    )

    lr.fit(x_train, y_train)

    y_pred = lr.predict(x_test)

    acc = accuracy_score(y_test, y_pred)

    results.append([c, acc])

results

[[0.01, 0.6991869918699187],
 [0.1, 0.8617886178861789],
 [1, 0.8617886178861789],
 [10, 0.8617886178861789],
 [100, 0.8617886178861789]]

### Finding:
- Performance stable for C ≥ 0.1.

### 5-fold cross-validation

In [130]:
from sklearn.model_selection import cross_val_score

lr = LogisticRegression(max_iter=1000, random_state=42)

scores = cross_val_score(
    lr,
    x_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

In [132]:
 scores

array([0.7979798 , 0.81632653, 0.81632653, 0.78571429, 0.7755102 ])

In [133]:
np.mean(scores)

np.float64(0.7983714698000413)

In [134]:
np.std(scores)

np.float64(0.016295788434948617)

### Findings:
- Logistic Regression achieved 86.18% accuracy on the held-out test set. 5-fold cross-validation produced a mean accuracy of 79.84% with a standard deviation of 1.63%, indicating stable performance across folds and suggesting that the single test-set result may be somewhat optimistic.

In [136]:
importance = pd.DataFrame({
    "Feature": x_train.columns,
    "Importance": rf.feature_importances_
})

importance.sort_values(
    by="Importance",
    ascending=False
)

,Feature,Importance
9,Credit_History,0.233219
5,ApplicantIncome,0.207221
7,LoanAmount,0.186270
6,CoapplicantIncome,0.107255
2,Dependents,0.048801
8,Loan_Amount_Term,0.047730
11,Property_Area_Semiurban,0.030721
3,Education,0.027685
1,Married,0.027466
12,Property_Area_Urban,0.023726


### GridSearchCV

In [146]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.001, 0.01, 0.1, 1, 10, 100],
    "penalty": ["l2"],
    "solver": ["lbfgs", "liblinear"]
}

grid = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(x_train, y_train)

,estimator,LogisticRegre...max_iter=1000)
,param_grid,"{'C': [0.001, 0.01, ...], 'penalty': ['l2'], 'solver': ['lbfgs', 'liblinear']}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [147]:
grid.best_params_

{'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}

In [148]:
grid.best_score_

np.float64(0.7983714698000413)

In [141]:
best_lr = grid.best_estimator_

In [143]:
y_pred = best_lr.predict(x_test)

In [149]:
accuracy_score(y_test, y_pred)

0.8617886178861789

### Observations
- Logistic Regression achieved the highest test accuracy.
- Decision Tree severely overfit the training data (100% train accuracy vs 77.24% test accuracy).
- Random Forest reduced overfitting but did not outperform Logistic Regression.
- Logistic Regression remained stable across different regularization strengths.
- Cross-validation showed consistent performance across folds.
### Final Model Chosen

- Logistic Regression

### Reasons
- Highest test accuracy among all evaluated models.
- Strong performance on the approval class.
- More interpretable than tree ensembles.
- Coefficients aligned with EDA findings.
- Stable during cross-validation.
- Hyperparameter tuning did not reveal a superior configuration.

In [153]:
import joblib
scaler = joblib.load("../models/scaler.pkl")

In [155]:
joblib.dump(model, "loan_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Files saved successfully")

['loan_model.pkl']

['scaler.pkl']